# Violin Prince Shorts Engine — cloud workspace

**No installation on your PC. No ZIP upload. No Google Drive folder.**

This notebook runs in Google's temporary Colab runtime. It analyses an exported violin performance, ranks musical moments, lets you audition them and renders the passage you select as a vertical Short.

**Privacy:** the video is uploaded to this temporary Colab session, not to GitHub or Google Drive. It can disappear when the runtime resets or disconnects.

**Current scope:** music-feature analysis and rendering only. YouTube retention access, Metricool publishing and recurring background runs are not connected yet.

## 1. Prepare the temporary cloud runtime

Run this cell first. FFmpeg and Python libraries are installed only inside this Colab session.

In [ ]:
import subprocess, sys, os, json, shutil, math
from pathlib import Path

subprocess.run(["bash", "-lc", "apt-get update -qq && apt-get install -y -qq ffmpeg"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "librosa>=0.11.0", "numpy>=1.26", "soundfile>=0.12", "pandas>=2.0"
], check=True)

print("Cloud runtime ready:")
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

## 2. Upload one of your violin videos

Choose an exported MP4, MOV, M4V, MKV or WebM. The upload is temporary and is not placed in GitHub or Google Drive. Only upload a performance you own or have permission to edit.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No video was uploaded.")

valid_ext = {".mp4", ".mov", ".m4v", ".mkv", ".webm"}
video_names = [name for name in uploaded if Path(name).suffix.lower() in valid_ext]
if not video_names:
    raise ValueError("Please upload MP4, MOV, M4V, MKV or WebM.")

VIDEO_PATH = Path("/content") / video_names[0]
print("Selected:", VIDEO_PATH.name)
print("Temporary upload size: {:.1f} MB".format(VIDEO_PATH.stat().st_size / (1024 * 1024)))

## 3. Analyse the musical moments

Choose a mode:
- `balanced`: general shortlist (recommended)
- `lyrical`: gives more weight to tonal movement and variation
- `energetic`: gives more weight to attacks, peaks and rhythmic activity

These are signal-feature heuristics, not an AI judgement of beauty. We will listen to the shortlist before choosing anything.

In [ ]:
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from IPython.display import Audio, Video, display

STYLE = "balanced"  # "balanced", "lyrical", or "energetic"
CANDIDATE_LIMIT = 8
MIN_SECONDS = 18
MAX_SECONDS = 45
SR = 22050
HOP = 512
WORK_DIR = Path("/content/vp_shorts_work")
WORK_DIR.mkdir(exist_ok=True)

AUDIO_PATH = WORK_DIR / "source_audio.wav"
subprocess.run([
    "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-i", str(VIDEO_PATH),
    "-vn", "-ac", "1", "-ar", str(SR), "-c:a", "pcm_s16le", str(AUDIO_PATH)
], check=True)

y, sr = sf.read(AUDIO_PATH, dtype="float32")
if y.ndim > 1:
    y = np.mean(y, axis=1)
duration = len(y) / sr
if duration < MIN_SECONDS:
    raise ValueError(f"Video audio is {duration:.1f}s; at least {MIN_SECONDS}s is needed.")

rms = librosa.feature.rms(y=y, frame_length=2048, hop_length=HOP)[0]
onset = librosa.onset.onset_strength(y=y, sr=sr, hop_length=HOP)
centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP)[0]
chroma = librosa.feature.chroma_stft(y=y, sr=sr, hop_length=HOP)
n = min(len(rms), len(onset), len(centroid), chroma.shape[1])
rms, onset, centroid, chroma = rms[:n], onset[:n], centroid[:n], chroma[:, :n]
times = librosa.frames_to_time(np.arange(n), sr=sr, hop_length=HOP)

tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr, hop_length=HOP, trim=False)
tempo_arr = np.asarray(tempo).reshape(-1)
bpm = float(tempo_arr[0]) if tempo_arr.size else 0.0
beats = librosa.frames_to_time(beat_frames, sr=sr, hop_length=HOP)

def robust_scale(values):
    values = np.asarray(values, dtype=float)
    lo, hi = np.nanpercentile(values, [10, 90])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-9:
        return np.zeros_like(values)
    return np.clip((values - lo) / (hi - lo), 0.0, 1.0)

def nearest_beat(t):
    if len(beats) == 0:
        return None
    return float(beats[int(np.argmin(np.abs(beats - t)))])

if STYLE == "energetic":
    weights = {"rms_mean": .22, "rms_variation": .12, "onset_mean": .22, "onset_peak": .18, "centroid_variation": .10, "chroma_motion": .06, "beat_alignment": .10}
elif STYLE == "lyrical":
    weights = {"rms_mean": .08, "rms_variation": .22, "onset_mean": .08, "onset_peak": .05, "centroid_variation": .12, "chroma_motion": .30, "beat_alignment": .15}
else:
    weights = {"rms_mean": .15, "rms_variation": .16, "onset_mean": .15, "onset_peak": .14, "centroid_variation": .12, "chroma_motion": .16, "beat_alignment": .12}

lengths = [x for x in [18.0, 24.0, 30.0, 36.0, 45.0] if x <= MAX_SECONDS and x <= duration]
raw = []
for length in lengths:
    for start in np.arange(0.0, max(0.01, duration - length) + 0.001, 3.0):
        end = min(float(start + length), duration)
        i0 = int(np.searchsorted(times, start, side="left"))
        i1 = int(np.searchsorted(times, end, side="right"))
        if i1 - i0 < 8:
            continue
        r, o, c, ch = rms[i0:i1], onset[i0:i1], centroid[i0:i1], chroma[:, i0:i1]
        chroma_motion = float(np.mean(np.linalg.norm(np.diff(ch, axis=1), axis=0))) if ch.shape[1] > 1 else 0.0
        start_beat, end_beat = nearest_beat(start), nearest_beat(end)
        period = 60.0 / bpm if bpm > 1 else 0.5
        beat_err = ((abs(start_beat - start) + abs(end_beat - end)) / max(period, .2)) if start_beat is not None and end_beat is not None else 0.0
        raw.append({
            "start": float(start), "end": float(end), "duration": float(end-start),
            "rms_mean": float(np.mean(r)), "rms_variation": float(np.std(r)),
            "onset_mean": float(np.mean(o)), "onset_peak": float(np.percentile(o, 90)),
            "centroid_variation": float(np.std(c)), "chroma_motion": chroma_motion,
            "beat_alignment": float(np.clip(1.0 - beat_err / 2.0, 0.0, 1.0))
        })

if not raw:
    raise RuntimeError("No candidate windows were found. Try a longer video.")

keys = list(weights)
scaled = {k: robust_scale([c[k] for c in raw]) for k in keys}
for i, c in enumerate(raw):
    c["score"] = 100.0 * sum(weights[k] * scaled[k][i] for k in keys)

ranked = sorted(raw, key=lambda c: c["score"], reverse=True)
chosen = []
for candidate in ranked:
    overlaps = False
    for existing in chosen:
        intersection = max(0.0, min(candidate["end"], existing["end"]) - max(candidate["start"], existing["start"]))
        union = max(candidate["end"], existing["end"]) - min(candidate["start"], existing["start"])
        if union > 0 and intersection / union > .35:
            overlaps = True
            break
    if not overlaps:
        chosen.append(candidate)
    if len(chosen) >= CANDIDATE_LIMIT:
        break

def fmt_time(seconds):
    return f"{int(seconds//60):02d}:{seconds%60:05.2f}"

for i, c in enumerate(chosen, 1):
    c["id"] = f"C{i:02d}"
    c["start_hms"] = fmt_time(c["start"])
    c["end_hms"] = fmt_time(c["end"])
    c["reason"] = f"Strong {STYLE} audio-feature match"

CANDIDATES = pd.DataFrame(chosen)
print(f"Analysis complete | Duration: {duration:.1f}s | Estimated tempo: {bpm:.1f} BPM | Mode: {STYLE}")
display(CANDIDATES[["id", "start_hms", "end_hms", "duration", "score", "reason"]].round({"duration": 1, "score": 1}))
print("Scores are ranking hints, not proof of musical quality. Audition each candidate.")

## 4. Audition the shortlist

Listen to the excerpts and check whether each one begins and ends at a natural musical point.

In [ ]:
def audition_candidate(candidate_id):
    row = CANDIDATES[CANDIDATES["id"].astype(str) == str(candidate_id)]
    if row.empty:
        raise ValueError(f"Candidate {candidate_id} was not found.")
    start = float(row.iloc[0]["start"])
    length = float(row.iloc[0]["duration"])
    preview_path = WORK_DIR / f"{candidate_id}_audition.mp3"
    subprocess.run([
        "ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-ss", str(start),
        "-i", str(VIDEO_PATH), "-t", str(length), "-vn", "-q:a", "4", str(preview_path)
    ], check=True)
    print(f"{candidate_id}: {row.iloc[0]['start_hms']} – {row.iloc[0]['end_hms']}")
    display(Audio(filename=str(preview_path)))

for candidate_id in CANDIDATES["id"].astype(str).tolist():
    audition_candidate(candidate_id)

## 5. Select one candidate

Change `CANDIDATE_ID` to the passage you prefer after listening.

In [ ]:
CANDIDATE_ID = "C01"  # change to C02, C03, etc.
selected_rows = CANDIDATES[CANDIDATES["id"].astype(str) == CANDIDATE_ID]
if selected_rows.empty:
    raise ValueError(f"Choose one of: {CANDIDATES['id'].tolist()}")
selected = selected_rows.iloc[0]
START_SECONDS = float(selected["start"])
END_SECONDS = float(selected["end"])
print(f"Selected {CANDIDATE_ID}: {selected['start_hms']}–{selected['end_hms']} ({selected['duration']:.1f}s)")

## 6. Render a vertical Short

The default blur-fit layout preserves the full original video frame rather than risking a crop that cuts off your violin, bow or hands.

In [ ]:
OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_{CANDIDATE_ID}_Short.mp4"
vf = (
    "split=2[bgsrc][fgsrc];"
    "[bgsrc]scale=270:480:force_original_aspect_ratio=increase,"
    "crop=270:480,boxblur=8:2,scale=1080:1920,eq=brightness=-0.20:saturation=0.85[bg];"
    "[fgsrc]scale=1080:1920:force_original_aspect_ratio=decrease,setsar=1[fg];"
    "[bg][fg]overlay=(W-w)/2:(H-h)/2,setsar=1[v]"
)
cmd = [
    "ffmpeg", "-hide_banner", "-y", "-ss", f"{START_SECONDS:.3f}", "-i", str(VIDEO_PATH),
    "-t", f"{END_SECONDS-START_SECONDS:.3f}", "-filter_complex", vf,
    "-map", "[v]", "-map", "0:a:0?", "-c:v", "libx264", "-preset", "ultrafast",
    "-crf", "22", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k",
    "-movflags", "+faststart", str(OUTPUT_VIDEO)
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("FFmpeg rendering failed.")
print("Rendered:", OUTPUT_VIDEO.name)
print("Size: {:.1f} MB".format(OUTPUT_VIDEO.stat().st_size / (1024 * 1024)))

## 7. Review the finished Short

Watch the whole clip. Check musical phrasing, ending, audio sync, violin/bow/hand visibility and overall image quality.

In [ ]:
display(Video(filename=str(OUTPUT_VIDEO), embed=True, width=360))

## 8. Optional: download the rendered MP4

Only run this if you want to download the finished Short. Otherwise the output remains in this temporary Colab session and may disappear when the session ends.

In [ ]:
from google.colab import files
files.download(str(OUTPUT_VIDEO))

## Current limitations

- This first version uses audio features only. It does not fetch YouTube audience-retention data.
- It does not publish to YouTube or schedule through Metricool.
- It does not run continuously in the background; Colab Free sessions are temporary and may disconnect.
- It does not generate AI music or full AI music videos.

The next stage is to verify candidate quality on a real Violin Prince performance, then add official YouTube Analytics and Metricool connections with your authorisation.